# 4: ML Model Training + REST API (Data Consumption)
**Pipeline Stage:** Train a purchase-intent classifier and expose it via a REST API.

This notebook demonstrates:
- Training a Random Forest classifier on processed data
- Model evaluation (accuracy, precision, recall, F1, ROC-AUC)
- Saving the model with `joblib`
- Building a **FastAPI** REST API to serve real-time predictions
- Running the API server directly from the notebook

In [2]:
%pip install pandas scikit-learn imbalanced-learn joblib fastapi uvicorn nest-asyncio pyarrow

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install requests

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
import sqlite3
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

BASE_DIR   = Path('..').resolve()
OUTPUT_DIR = BASE_DIR / 'outputs'
DB_PATH    = OUTPUT_DIR / 'pipeline.db'
MODEL_PATH = OUTPUT_DIR / 'purchase_intent_model.joblib'
print("Ready.")

Ready.


## 4.1  Load Processed Data

In [5]:
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql('SELECT * FROM processed_sessions', conn)
conn.close()

X = df.drop(columns=['Revenue'])
y = df['Revenue']
print(f"Features: {X.shape[1]}  |  Samples: {len(X):,}  |  Class balance: {y.mean():.1%} positive")
X.head(2)

Features: 33  |  Samples: 20,594  |  Class balance: 50.0% positive


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,VisitorType_enc,Month_Dec,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep
0,0,-0.460019,0,-0.246257,1,-0.628793,3.969402,3.434394,-0.318962,0.0,...,2,0,1,0,0,0,0,0,0,0
1,0,-0.460019,0,-0.246257,2,-0.595451,-0.450137,1.268054,-0.318962,0.0,...,2,0,1,0,0,0,0,0,0,0


In [6]:
df.tail()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,...,Month_Dec,Month_Feb,Month_Jul,Month_June,Month_Mar,Month_May,Month_Nov,Month_Oct,Month_Sep,Revenue
20589,5,0.401121,1,0.218218,60,0.295337,-0.436804,-0.493413,-0.184420,0.0,...,0,0,0,0,0,0,1,0,0,1
20590,8,1.119347,2,0.189016,170,2.722912,-0.299789,-0.519406,-0.092451,0.0,...,0,0,0,0,0,0,1,0,0,1
20591,12,0.935332,2,0.058864,161,1.437346,-0.321863,-0.512470,0.521105,0.0,...,0,0,0,0,0,0,1,0,0,1
20592,4,-0.092859,0,-0.246257,43,0.225698,-0.196219,-0.232826,-0.157628,0.0,...,0,0,0,0,0,0,0,0,0,1
20593,3,-0.144152,0,-0.246257,139,1.414886,-0.400694,-0.643601,0.636597,0.0,...,0,0,0,0,0,0,1,0,0,1


## 4.2  Train/Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

Train: 16,475  |  Test: 4,119


## 4.3  Model Comparison

In [8]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    results.append({'Model': name, 'CV_ROC_AUC_Mean': scores.mean(), 'CV_ROC_AUC_Std': scores.std()})
    print(f"{name:25s}  AUC={scores.mean():.4f} ± {scores.std():.4f}")

df_results = pd.DataFrame(results).sort_values('CV_ROC_AUC_Mean', ascending=False)
df_results

Logistic Regression        AUC=0.9652 ± 0.0019
Random Forest              AUC=0.9829 ± 0.0013
Gradient Boosting          AUC=0.9739 ± 0.0013


,Model,CV_ROC_AUC_Mean,CV_ROC_AUC_Std
1,Random Forest,0.982888,0.001336
2,Gradient Boosting,0.973909,0.001343
0,Logistic Regression,0.965235,0.001857


## 4.4  Train Best Model (Random Forest) & Evaluate

In [9]:
best_model = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)

y_pred  = best_model.predict(X_test)
y_prob  = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['No Purchase','Purchase']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

              precision    recall  f1-score   support

 No Purchase       0.95      0.91      0.93      2060
    Purchase       0.91      0.95      0.93      2059

    accuracy                           0.93      4119
   macro avg       0.93      0.93      0.93      4119
weighted avg       0.93      0.93      0.93      4119

ROC-AUC: 0.9806


In [10]:
# Feature importances
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns)\
             .sort_values(ascending=False).head(15)
print("Top 15 Feature Importances:")
print(feat_imp.round(4).to_string())

Top 15 Feature Importances:
PageValues                 0.3401
IsHighValue                0.1551
ExitRates                  0.0502
EngagementScore            0.0448
MonthNum                   0.0339
ProductRelated_Duration    0.0310
Administrative_Duration    0.0309
AvgDurationPerPage         0.0294
BounceRates                0.0287
TotalDuration              0.0275
ProductRelated             0.0255
TotalPages                 0.0238
ProductFocusRatio          0.0233
Administrative             0.0182
VisitorType_enc            0.0165


## 4.5  Save Model

In [11]:
# Save model + feature column names together so the API can reconstruct inputs
model_artifact = {
    'model':    best_model,
    'features': list(X.columns),
}
joblib.dump(model_artifact, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}  ({MODEL_PATH.stat().st_size // 1024} KB)")

Model saved to C:\Stuffs\University Of New Haven\MSDS\Second Semester\Distributed and Scalable Data Engineering\Final\Project\Scripts\pipeline_project\pipeline\outputs\purchase_intent_model.joblib  (25016 KB)


## 4.6  FastAPI REST API
Run the cell below to start a local prediction server on **http://127.0.0.1:8000**.

- `GET /`               → health check
- `POST /predict`       → single session prediction  
- `POST /predict_batch` → batch predictions
- `GET /docs`           → interactive Swagger UI

In [12]:
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional

nest_asyncio.apply()  # Allow uvicorn inside Jupyter

# Load model artifact
artifact = joblib.load(MODEL_PATH)
clf      = artifact['model']
features = artifact['features']

app = FastAPI(
    title='Purchase Intent Prediction API',
    description='Predicts whether an online shopping session will result in a purchase.',
    version='1.0'
)

# ── Input Schema ───────────────────────────────────────────────────────────────
class SessionInput(BaseModel):
    Administrative:             int   = Field(0,   description="Number of admin pages visited")
    Administrative_Duration:    float = Field(0.0, description="Time on admin pages (s)")
    Informational:              int   = Field(0)
    Informational_Duration:     float = Field(0.0)
    ProductRelated:             int   = Field(10)
    ProductRelated_Duration:    float = Field(500.0)
    BounceRates:                float = Field(0.02)
    ExitRates:                  float = Field(0.05)
    PageValues:                 float = Field(20.0)
    SpecialDay:                 float = Field(0.0)
    OperatingSystems:           int   = Field(2)
    Browser:                    int   = Field(2)
    Region:                     int   = Field(1)
    TrafficType:                int   = Field(2)
    Weekend:                    int   = Field(0, description="1=Weekend, 0=Weekday")
    VisitorType_enc:            int   = Field(1, description="0=New, 1=Returning, 2=Other")
    MonthNum:                   int   = Field(11, description="1=Jan .. 12=Dec")

class PredictionOutput(BaseModel):
    will_purchase:       bool
    purchase_probability: float
    confidence:          str

# ── Helpers ────────────────────────────────────────────────────────────────────
def build_feature_vector(session: dict) -> pd.DataFrame:
    """Convert raw session input to the engineered feature vector."""
    s = session
    total_pages    = s['Administrative'] + s['Informational'] + s['ProductRelated']
    total_duration = (s['Administrative_Duration'] + s['Informational_Duration'] +
                      s['ProductRelated_Duration'])
    row = {f: 0 for f in features}  # zero-fill all OHE month columns
    row.update({
        'Administrative':             s['Administrative'],
        'Administrative_Duration':    s['Administrative_Duration'],
        'Informational':              s['Informational'],
        'Informational_Duration':     s['Informational_Duration'],
        'ProductRelated':             s['ProductRelated'],
        'ProductRelated_Duration':    s['ProductRelated_Duration'],
        'BounceRates':                s['BounceRates'],
        'ExitRates':                  s['ExitRates'],
        'PageValues':                 s['PageValues'],
        'SpecialDay':                 s['SpecialDay'],
        'OperatingSystems':           s['OperatingSystems'],
        'Browser':                    s['Browser'],
        'Region':                     s['Region'],
        'TrafficType':                s['TrafficType'],
        'Weekend':                    s['Weekend'],
        'VisitorType_enc':            s['VisitorType_enc'],
        'MonthNum':                   s['MonthNum'],
        'TotalPages':                 total_pages,
        'TotalDuration':              total_duration,
        'AvgDurationPerPage':         total_duration / (total_pages + 1),
        'ProductFocusRatio':          s['ProductRelated'] / (total_pages + 1),
        'EngagementScore':            1 - (s['BounceRates'] + s['ExitRates']) / 2,
        'IsHighValue':                1 if s['PageValues'] > 5.0 else 0,
        'IsQ4':                       1 if s['MonthNum'] in [10,11,12] else 0,
    })
    return pd.DataFrame([row])[features]

# ── Endpoints ──────────────────────────────────────────────────────────────────
@app.get('/')
def health():
    return {'status': 'ok', 'model': 'Random Forest v1', 'features': len(features)}

@app.post('/predict', response_model=PredictionOutput)
def predict(session: SessionInput):
    try:
        X_input = build_feature_vector(session.dict())
        prob    = float(clf.predict_proba(X_input)[0, 1])
        will_buy = prob >= 0.5
        confidence = 'high' if abs(prob - 0.5) > 0.3 else 'medium' if abs(prob - 0.5) > 0.15 else 'low'
        return PredictionOutput(
            will_purchase=will_buy,
            purchase_probability=round(prob, 4),
            confidence=confidence
        )
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post('/predict_batch')
def predict_batch(sessions: List[SessionInput]):
    results = []
    for s in sessions:
        X_input = build_feature_vector(s.dict())
        prob = float(clf.predict_proba(X_input)[0, 1])
        results.append({'purchase_probability': round(prob, 4), 'will_purchase': prob >= 0.5})
    return {'predictions': results, 'count': len(results)}

print("API defined. Run the next cell to start the server on http://127.0.0.1:8000")
print("Interactive docs: http://127.0.0.1:8000/docs")

API defined. Run the next cell to start the server on http://127.0.0.1:8000
Interactive docs: http://127.0.0.1:8000/docs


In [13]:
# ── Start server in a background thread ────────────────────────────────────────
# The server will run until the notebook kernel is stopped.
def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

import time; time.sleep(1.5)  # let the server start

# Quick test
import requests
resp = requests.get('http://127.0.0.1:8000/')
print("Health:", resp.json())

test_payload = {
    "Administrative": 0, "Administrative_Duration": 0,
    "Informational": 0,  "Informational_Duration": 0,
    "ProductRelated": 15,"ProductRelated_Duration": 800,
    "BounceRates": 0.01, "ExitRates": 0.04,
    "PageValues": 35.0,  "SpecialDay": 0.0,
    "OperatingSystems": 2, "Browser": 2, "Region": 1,
    "TrafficType": 2, "Weekend": 1, "VisitorType_enc": 1, "MonthNum": 11
}
resp = requests.post('http://127.0.0.1:8000/predict', json=test_payload)
print("Prediction:", resp.json())

Health: {'status': 'ok', 'model': 'Random Forest v1', 'features': 33}
Prediction: {'will_purchase': True, 'purchase_probability': 0.7552, 'confidence': 'medium'}


## Summary
- **Best model**: Random Forest (200 trees)
- **API**: FastAPI on `http://127.0.0.1:8000`
- **Model saved to**: `outputs/purchase_intent_model.joblib`
- **Swagger UI**: `http://127.0.0.1:8000/docs`
